# Test the S3 OLCI L1 processor

Implementation through following tasks:
- RSPY-1003: https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1003
- RSPY-1000: https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-1000

In [ ]:
# Experimental DPR processor configuration, used only for testing.
# See: https://github.com/RS-PYTHON/rs-dpr-service/blob/develop/rs_dpr_service/utils/settings.py
experimental_config = {
    "local_cluster": {
        "enabled": False, # Use False to disable
        "n_workers": 4,
        "memory_limit": "58GiB",
    },
    "local_files": {
        "local_dir": None, #"/tmp/data", # Use None to disable
        "overwrite_input": False,
        "upload_output": True,
    },
}

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo()
# Reload the global vars again
from resources.utils import *  

from resources.dask_clusters.dask_main_env import *
await init_dask_cluster_s3olci()

In [ ]:
# Other imports
import os.path as osp
from IPython.display import JSON
from resources.dpr_utils import DprDemo
from rs_client.ogcapi.dpr_client import DprProcessor

## Read the tasktable

<div class="alert alert-block alert-warning">
Note: for now the S3OLCI processor returns dummy values that are not usable.
</div>

In [ ]:
tasktable: dict = dpr_client.get_process(DprProcessor.S3L1OLCI.value, cluster_info_eopf)
print(f"Tasktable for {DprProcessor.S3L1OLCI.value!r}:")
display(JSON(tasktable))

## Init environment for the processors

In [ ]:
# Init DPR processor demo
dpr = DprDemo(
    owner_id=OWNER_ID, 
    dpr_client=dpr_client,
    local_config_dir="./config"
)

await dpr.init(local_secrets_file="./config/secrets.json")

# Arguments for S3 L0
s3_args = {
    "process": DprProcessor.S3L1OLCI.value,
    "cluster_info": cluster_info_eopf,
    "payload_subpath": "s3l1olci/basic_payload.yaml",
    "experimental_config": experimental_config,
}

## Run the S3 OLCI L1 processor

In [ ]:
# Run S3 full data
# Careful here "s3_output_dir" is for s3 bucket and not s3 data
if os.getenv("RSPY_FROM_CICD") != "1":
    s3_output_dir = osp.join(dpr.s3_output_dir, "s3olci", "l1")
    await dpr.run_process(
        **s3_args,
        s3_output_dir = s3_output_dir,
        s3_report_dir = osp.join(dpr.s3_report_dir, "s3olci", "l1"),
        # Payload env vars
        OUTPUT_DIR = s3_output_dir,
        SHORT_SUFFIX="",
    )